In [9]:
import pandas as pd
import numpy as np
import glob
import os
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

# --- PART 1: REUSE ASHRAE FUNCTION ---
def calculate_ashrae_metrics(y_true, y_pred, split_name, zone, load, model_name):
    if len(y_true) == 0: return None
    
    n = len(y_true)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    mean_y = np.mean(y_true)
    
    cv_rmse = (rmse / mean_y) * 100 if mean_y != 0 else 0
    nmbe = (np.sum(y_true - y_pred) / (n * mean_y)) * 100 if mean_y != 0 else 0
    ashrae_pass = (cv_rmse <= 30) and (-10 <= nmbe <= 10)
    
    return {
        'zone': zone, 'load': load, 'model': model_name, 'split': split_name,
        'n': n, 'MAE': mae, 'RMSE': rmse, 'CV_RMSE': cv_rmse, 'NMBE': nmbe, 
        'SHRAE_pass': ashrae_pass
    }

# --- PART 2: CALCULATE BLEND MODEL ---
print("Calculating Blend Model...")

# 1. Load your GBM Master file
df_gbm = pd.read_csv('All_Zones_GBM_predictions.csv')
# Rename the prediction column to avoid confusion during merge
df_gbm = df_gbm.rename(columns={'y_pred': 'y_pred_gbm'}) 

# 2. Find and load all Chronos predictions from Ludovica
# This automatically searches all subfolders for csv files containing 'pre' in the name
chronos_files = glob.glob('**/*pre*.csv', recursive=True)
chronos_list = []

for file in chronos_files:
    # Check if it's actually a Chronos file and not your GBM file
    if 'GBM' not in file:
        df_temp = pd.read_csv(file)
        chronos_list.append(df_temp)

if not chronos_list:
    print("Error: No Chronos files found!")
else:
    # Concatenate all Chronos files together
    df_chronos = pd.concat(chronos_list, ignore_index=True)
    
    # Find the column containing the predictions and rename it
    pred_col = [col for col in df_chronos.columns if 'pred' in col.lower()][0]
    df_chronos = df_chronos.rename(columns={pred_col: 'y_pred_chronos'})
    
    # Ensure timestamps have the same format
    df_gbm['timestamp_utc'] = pd.to_datetime(df_gbm['timestamp_utc'])
    df_chronos['timestamp_utc'] = pd.to_datetime(df_chronos['timestamp_utc'])

    # 3. Merge the two models based on time, zone, and load
    df_blend = pd.merge(df_gbm, df_chronos[['timestamp_utc', 'zone', 'load_type', 'y_pred_chronos']], 
                        on=['timestamp_utc', 'zone', 'load_type'], 
                        how='inner')

    # 4. Calculate the Blend! (Simple 50/50 average)
    df_blend['y_pred_blend'] = (df_blend['y_pred_gbm'] + df_blend['y_pred_chronos']) / 2

    blend_metrics = []
    
    # 5. Calculate the new ASHRAE scores for the Blend model
    for zone in df_blend['zone'].unique():
        for load in df_blend['load_type'].unique():
            df_subset = df_blend[(df_blend['zone'] == zone) & (df_blend['load_type'] == load)]
            if df_subset.empty: continue
                
            # Filter per year (split)
            for split_name in ['test_2023', 'test_2025']:
                year = int(split_name[-4:])
                df_year = df_subset[df_subset['timestamp_utc'].dt.year == year]
                
                if not df_year.empty:
                    metrics = calculate_ashrae_metrics(
                        df_year['y_true'], df_year['y_pred_blend'], split_name, zone, load, 'GBM_Chronos_Blend'
                    )
                    if metrics: blend_metrics.append(metrics)

    # 6. Save the Blend results
    df_blend_metrics = pd.DataFrame(blend_metrics)
    df_blend_metrics.to_excel('All_Zones_BLEND_ASHRAE_metrics.xlsx', index=False)
    print("Blend metrics calculated and saved.")

    # --- PART 3: CREATE COMPARISON TABLE ---
    print("Generating comparison table...")
    gbm_df = pd.read_excel("All_Zones_GBM_ASHRAE_metrics.xlsx")
    
    # Find all individual Chronos result files
    chronos_results_files = glob.glob("*_chronos_bolt_base_results.xlsx")
    chronos_results_dfs = [pd.read_excel(f) for f in chronos_results_files]
    df_chronos_results = pd.concat(chronos_results_dfs, ignore_index=True)

    # Rename columns for the merge
    df_chronos_results = df_chronos_results.rename(columns={'load_type': 'load', 'CV_RMSE_%': 'CV_RMSE_chronos', 'ASHRAE_pass': 'pass_chronos'})
    gbm_df = gbm_df.rename(columns={'CV_RMSE': 'CV_RMSE_gbm', 'SHRAE_pass': 'pass_gbm'})
    df_blend_metrics = df_blend_metrics.rename(columns={'CV_RMSE': 'CV_RMSE_blend', 'SHRAE_pass': 'pass_blend'})

    # Merge everything together
    merged_df = pd.merge(gbm_df[['zone', 'load', 'split', 'CV_RMSE_gbm', 'pass_gbm']], 
                         df_chronos_results[['zone', 'load', 'split', 'CV_RMSE_chronos', 'pass_chronos']], 
                         on=['zone', 'load', 'split'], how='inner')

    final_comparison = pd.merge(merged_df, 
                                df_blend_metrics[['zone', 'load', 'split', 'CV_RMSE_blend', 'pass_blend']], 
                                on=['zone', 'load', 'split'], how='inner')

    # Round and sort
    final_comparison['CV_RMSE_gbm'] = final_comparison['CV_RMSE_gbm'].round(1)
    final_comparison['CV_RMSE_chronos'] = final_comparison['CV_RMSE_chronos'].round(1)
    final_comparison['CV_RMSE_blend'] = final_comparison['CV_RMSE_blend'].round(1)
    final_comparison = final_comparison.sort_values(by=['zone', 'load', 'split']).reset_index(drop=True)

    # Save to Excel
    final_comparison.to_excel("Model_Comparison_ASHRAE.xlsx", index=False)
    
    print("\n--- SUCCESS! ---")
    print("The comparison has been successfully saved as 'Model_Comparison_ASHRAE.xlsx'")

Calculating Blend Model...
Blend metrics calculated and saved.
Generating comparison table...

--- SUCCESS! ---
The comparison has been successfully saved as 'Model_Comparison_ASHRAE.xlsx'
